# 📖 Quickstart Guide — YOLOv8x Retail Product Pricing-Recognition

This notebook explains the repository layout and shows you how to:
1. Set up the environment
2. Use the RPC dataset **or** plug in your own COCO-format dataset
3. Configure training
4. Run inference on images and videos

For the full end-to-end RPC training pipeline, open **`01_train_rpc.ipynb`**.

## Table of Contents
- [1. Repository Layout](#layout)
- [2. Environment Setup](#env)
- [3. Dataset Options](#data)
  - [3a. RPC Dataset (Kaggle)](#rpc)
  - [3b. Custom COCO Dataset](#custom-coco)
  - [3c. Custom Price Catalogue](#custom-price)
- [4. Training Configuration](#train-cfg)
- [5. Training](#train)
- [6. Image Inference](#img-infer)
- [7. Video Inference](#vid-infer)
- [8. Export & Download](#export)

---
## 1. Repository Layout <a id='layout'></a>

```
YOLOv8x Product Pricing-Recognition/
│
├── notebooks/
│   ├── 00_quickstart.ipynb      ← YOU ARE HERE
│   └── 01_train_rpc.ipynb       ← Full RPC training pipeline
│
├── src/
│   ├── __init__.py
│   ├── dataset.py               ← Download + COCO→YOLO conversion + subset sampling
│   ├── pricing.py               ← Price catalogue (load / save / assign / receipt)
│   ├── train.py                 ← Training entry point (notebook or CLI)
│   └── inference.py             ← Image and video inference with price overlay
│
├── configs/
│   └── train_config.yaml        ← All training hyper-parameters
│
├── data/
│   └── price_catalog_default.json   ← Template price catalogue (40 classes)
│
└── requirements.txt
```

After training, Ultralytics creates:
```
runs/
└── yolov8x_rpc/
    ├── weights/
    │   ├── best.pt              ← Use this for inference
    │   └── last.pt
    └── results.csv              ← Training metrics per epoch
```

---
## 2. Environment Setup <a id='env'></a>

In [ ]:
# Verify GPU
!nvidia-smi

# Install dependencies
!pip install -q ultralytics kagglehub pycocotools opencv-python-headless tqdm

import torch
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")

In [ ]:
# ── Mount the repo so Python can import src/ ──────────────────────
# Option A: clone from GitHub (replace URL with yours)
# !git clone https://github.com/youruser/yolov8x-retail-pricing /content/repo
# import sys; sys.path.insert(0, '/content/repo')

# Option B: upload the project folder to Colab, then:
import sys, os
REPO = '/content/YOLOv8x Product Pricing-Recognition'   # ← adjust if needed
sys.path.insert(0, REPO)
os.chdir(REPO)
print('Working dir:', os.getcwd())

In [ ]:
from src.dataset   import RPCDatasetConverter
from src.pricing   import PriceCatalog
from src.train     import train, evaluate
from src.inference import ProductDetector
print('All modules imported successfully.')

---
## 3. Dataset Options <a id='data'></a>

The pipeline accepts **any COCO-format dataset**.  
Choose one of the three options below.

### 3a. RPC Dataset from Kaggle <a id='rpc'></a>

**Prerequisites** — add your Kaggle credentials to Colab Secrets (🔑 icon on the left sidebar):
- Secret name `KAGGLE_USERNAME` → your Kaggle username
- Secret name `KAGGLE_KEY`      → your Kaggle API key

Get your key at: https://www.kaggle.com/settings → API → Create New Token

In [ ]:
from google.colab import userdata
import os
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')

In [ ]:
converter = RPCDatasetConverter(output_dir='/content/rpc_subset')

# Step 1 — Download (~4 GB, takes 2–5 min)
converter.download_rpc()

# Step 2 — Convert + sample 10% of images
#   Change subset_frac=1.0 to use the full dataset
converter.convert(subset_frac=0.10)

# Step 3 — Write dataset YAML
YAML_PATH = converter.write_yaml()
print('Dataset ready:', YAML_PATH)

### 3b. Custom COCO Dataset <a id='custom-coco'></a>

If you have your own dataset already annotated in COCO format, skip the
`download_rpc()` call and pass paths directly to `convert()`.

In [ ]:
# ── CUSTOM DATASET — fill in your paths ──────────────────────────
# (run this cell instead of the RPC cell above)

converter = RPCDatasetConverter(output_dir='/content/custom_subset')

converter.convert(
    train_ann_path = '/content/my_dataset/annotations/train.json',
    train_img_dir  = '/content/my_dataset/images/train',
    val_ann_path   = '/content/my_dataset/annotations/val.json',
    val_img_dir    = '/content/my_dataset/images/val',
    subset_frac    = 1.0,   # use all data
)

YAML_PATH = converter.write_yaml()
print('Dataset ready:', YAML_PATH)

### 3c. Custom Price Catalogue <a id='custom-price'></a>

After conversion, a `price_catalog.json` is auto-generated with keyword-matched
prices. You can override it two ways:

**Option 1 — Edit `data/price_catalog_default.json`** (for the template 40-class catalogue)

**Option 2 — Build from a plain dict:**

In [ ]:
# Build a catalogue from scratch
catalog = PriceCatalog.from_custom({
    'cola_330ml':   3.50,
    'chips_100g':   5.50,
    'mineral_water': 2.00,
    'chocolate_80g': 9.50,
})
catalog.save('/content/rpc_subset/price_catalog.json')

# Or load an existing one
catalog = PriceCatalog.from_json('/content/rpc_subset/price_catalog.json')
print(catalog)
print('Class names:', catalog.class_names()[:5], '...')

---
## 4. Training Configuration <a id='train-cfg'></a>

All hyper-parameters live in **`configs/train_config.yaml`**.  
Key settings to tune:

| Parameter | Default | When to change |
|-----------|---------|----------------|
| `epochs`  | 50      | Increase to 100–200 for full dataset |
| `batch`   | 16      | Lower to 8 on T4; raise to 32 on A100 |
| `imgsz`   | 640     | Raise to 1280 for small-object accuracy |
| `weights` | yolov8x.pt | Swap for `yolov8n/s/m/l.pt` for speed |
| `patience`| 15      | Early stopping patience in epochs |

Edit the YAML in place or override via keyword args when calling `train()`.

In [ ]:
# Preview current config
with open('configs/train_config.yaml') as f:
    print(f.read())

---
## 5. Training <a id='train'></a>

In [ ]:
from src.train import train

trained_model = train(
    data        = YAML_PATH,
    config_path = 'configs/train_config.yaml',
    # Override individual params without editing the YAML:
    # epochs = 10,
    # batch  = 8,
)
print('Done. Model:', trained_model)

In [ ]:
from src.train import evaluate

metrics = evaluate(
    model_path = 'runs/yolov8x_rpc/weights/best.pt',
    data       = YAML_PATH,
)
print(metrics)

---
## 6. Image Inference <a id='img-infer'></a>

In [ ]:
import matplotlib.pyplot as plt
from src.inference import ProductDetector

detector = ProductDetector(
    model_path   = 'runs/yolov8x_rpc/weights/best.pt',
    catalog_path = '/content/rpc_subset/price_catalog.json',
    conf = 0.30,
    iou  = 0.45,
)

# Run on any image
IMAGE_PATH = '/content/rpc_subset/images/val/'  # or your own image path
from pathlib import Path
sample = next(Path(IMAGE_PATH).glob('*.jpg'))

annotated, receipt = detector.infer_image(sample)

plt.figure(figsize=(12, 8))
plt.imshow(annotated)
plt.axis('off')
plt.title(f"Detected {sum(receipt['counts'].values())} items  |  Total ¥{receipt['total']:.2f}")
plt.tight_layout()
plt.show()

print('\nReceipt:')
for name, count, subtotal in receipt['lines']:
    print(f'  {name:30s} x{count}  ¥{subtotal:.2f}')
print(f'  {"TOTAL":30s}     ¥{receipt["total"]:.2f}')

---
## 7. Video Inference <a id='vid-infer'></a>

Upload a `.mp4` via the Files panel (📁 on the left) then set `VIDEO_INPUT`.

In [ ]:
VIDEO_INPUT  = '/content/retail_checkout.mp4'   # ← your file
VIDEO_OUTPUT = '/content/retail_output.mp4'

from pathlib import Path
if not Path(VIDEO_INPUT).exists():
    print('Video not found. Upload it via the Files panel on the left, then re-run.')
else:
    totals = detector.process_video(
        video_path  = VIDEO_INPUT,
        output_path = VIDEO_OUTPUT,
        conf        = 0.30,
        skip_frames = 1,   # 2–3 for faster processing
    )
    print(f"Session total: ¥{totals['total']:.2f}")

    # Download the annotated video
    from google.colab import files
    files.download(VIDEO_OUTPUT)

---
## 8. Export & Download <a id='export'></a>

In [ ]:
import zipfile
from ultralytics import YOLO
from google.colab import files

# Export to ONNX for deployment
model = YOLO('runs/yolov8x_rpc/weights/best.pt')
model.export(format='onnx', imgsz=640, dynamic=True, opset=17)

# Bundle weights + catalogue
with zipfile.ZipFile('/content/yolov8x_rpc_weights.zip', 'w') as zf:
    zf.write('runs/yolov8x_rpc/weights/best.pt',    'best.pt')
    zf.write('/content/rpc_subset/price_catalog.json', 'price_catalog.json')
    zf.write('/content/rpc_subset/dataset.yaml',       'dataset.yaml')

files.download('/content/yolov8x_rpc_weights.zip')